In [5]:
# %%
import pandas as pd

df = pd.read_csv("/home/thinkpad/Documents/BrototypeStudying/Paper two/week 8/code/data/raw_employee_data.csv")

# %%
# Drop rows with no target — can't train/evaluate without salary
df = df.dropna(subset="salary")

# %%
# Drop only truly non-predictive columns
df = df.drop(columns=["employee_id", "name", "email", "phone", "join_date", "notes"])

# %%
from sklearn.model_selection import train_test_split

x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# %%
# --- Clean age ---
median_age = x_train["age"].median()

x_train.loc[x_train["age"] > 100, "age"] = median_age
x_test.loc[x_test["age"] > 100, "age"] = median_age

x_train.loc[x_train["age"] < 18, "age"] = median_age
x_test.loc[x_test["age"] < 18, "age"] = median_age

x_train["age"] = x_train["age"].fillna(median_age)
x_test["age"] = x_test["age"].fillna(median_age)

# %%
# --- Clean department ---
department_mode = x_train["department"].mode()[0]
x_train["department"] = x_train["department"].fillna(department_mode)
x_test["department"] = x_test["department"].fillna(department_mode)

x_train["department"] = x_train["department"].str.strip().str.lower()
x_test["department"] = x_test["department"].str.strip().str.lower()

dept_map = {
    "suport": "support",
    "i.t.": "it",
    "human resources": "hr",
    "h.r.": "hr",
    "markting": "marketing",
    "ops": "operations"
}
x_train["department"] = x_train["department"].replace(dept_map)
x_test["department"] = x_test["department"].replace(dept_map)

# %%
# --- Clean city ---
city_mode = x_train["city"].mode()[0]
x_train["city"] = x_train["city"].fillna(city_mode)
x_test["city"] = x_test["city"].fillna(city_mode)

x_train["city"] = x_train["city"].str.strip().str.lower()
x_test["city"] = x_test["city"].str.strip().str.lower()

city_map = {"nyc": "new york"}
x_train["city"] = x_train["city"].replace(city_map)
x_test["city"] = x_test["city"].replace(city_map)

# %%
# --- Clean years_experience ---
median_exp = x_train["years_experience"].median()
x_train["years_experience"] = x_train["years_experience"].fillna(median_exp)
x_test["years_experience"] = x_test["years_experience"].fillna(median_exp)

invalid_train = x_train["years_experience"] > (x_train["age"] - 18)
x_train.loc[invalid_train, "years_experience"] = (x_train.loc[invalid_train, "age"] - 18).clip(lower=0)

invalid_test = x_test["years_experience"] > (x_test["age"] - 18)
x_test.loc[invalid_test, "years_experience"] = (x_test.loc[invalid_test, "age"] - 18).clip(lower=0)

# %%
# --- Clean remote_work ---
mode_remote = x_train["remote_work"].mode()[0]
x_train["remote_work"] = x_train["remote_work"].fillna(mode_remote)
x_test["remote_work"] = x_test["remote_work"].fillna(mode_remote)

# %%
# --- Clean performance_rating ---
median_perf = x_train["performance_rating"].median()
x_train["performance_rating"] = x_train["performance_rating"].fillna(median_perf)
x_test["performance_rating"] = x_test["performance_rating"].fillna(median_perf)

# %%
# --- Clean education ---
edu_mode = x_train["education"].mode()[0]
x_train["education"] = x_train["education"].fillna(edu_mode)
x_test["education"] = x_test["education"].fillna(edu_mode)

x_train["education"] = x_train["education"].str.strip().str.lower()
x_test["education"] = x_test["education"].str.strip().str.lower()

# Fix duplicate category: "bachelor" vs "bachelors"
edu_map = {
    "bachelor": "bachelors"
}
x_train["education"] = x_train["education"].replace(edu_map)
x_test["education"] = x_test["education"].replace(edu_map)

print(x_train["education"].unique())  # confirm clean now

# %%
# --- Clean gender ---
gender_mode = x_train["gender"].mode()[0]
x_train["gender"] = x_train["gender"].fillna(gender_mode)
x_test["gender"] = x_test["gender"].fillna(gender_mode)

x_train["gender"] = x_train["gender"].str.strip().str.lower()
x_test["gender"] = x_test["gender"].str.strip().str.lower()

# %%
# --- One-hot encode all categorical columns ---
cat_cols = ["department", "city", "education", "gender"]

x_train = pd.get_dummies(x_train, columns=cat_cols, drop_first=True).astype(int)
x_test = pd.get_dummies(x_test, columns=cat_cols, drop_first=True).astype(int)

# Align columns in case a category appears only in train or only in test
x_train, x_test = x_train.align(x_test, join="left", axis=1, fill_value=0)


# %%
# --- Clean salary (target) ---
y_train = y_train.astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)
y_test = y_test.astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)

median_salary = y_train.median()
y_train[y_train > 200000] = median_salary
y_test[y_test > 200000] = median_salary

# %%
# --- Scale numeric columns ---
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
cols_to_scale = ["age", "years_experience", "performance_rating"]

x_train[cols_to_scale] = scaler.fit_transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler.transform(x_test[cols_to_scale])

# %%
# --- Train final model: Linear Regression ---
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

model = LinearRegression()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print("Linear Regression")
print("MAE :", mean_absolute_error(y_test, pred))
print("R2  :", r2_score(y_test, pred))

# %%
# --- Check coefficients (which features push salary up/down) ---
coefficients = pd.Series(model.coef_, index=x_train.columns).sort_values(ascending=False)
print("\nTop positive influences on salary:")
print(coefficients.head(10))
print("\nTop negative influences on salary:")
print(coefficients.tail(10))

<StringArray>
['phd', 'master', 'bachelors', 'high school']
Length: 4, dtype: str
Linear Regression
MAE : 4979.483131565532
R2  : 0.7973426342057656

Top positive influences on salary:
education_phd       18714.975744
education_master     8571.865227
years_experience     7080.126399
city_denver          2430.168649
city_chicago         1573.421175
gender_male          1555.035660
gender_m             1507.122450
gender_female        1426.070398
city_seattle         1291.464454
city_new york        1240.941797
dtype: float64

Top negative influences on salary:
remote_work                111.794895
city_miami                -562.331433
education_high school    -4958.781024
department_it            -6638.683697
department_finance       -8922.207918
department_marketing    -21766.370927
department_sales        -23634.845298
department_operations   -26669.054369
department_hr           -27318.371091
department_support      -30992.637935
dtype: float64


In [1]:
# %%
import pandas as pd

df = pd.read_csv("/home/thinkpad/Documents/BrototypeStudying/Paper two/week 8/code/data/raw_employee_data.csv")

# %%
df = df.dropna(subset="salary")

# %%
df = df.drop(columns=["employee_id", "name", "email", "phone", "join_date", "notes"])

# %%
from sklearn.model_selection import train_test_split

x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# %%
# --- Clean age ---
median_age = x_train["age"].median()

x_train.loc[x_train["age"] > 100, "age"] = median_age
x_test.loc[x_test["age"] > 100, "age"] = median_age

x_train.loc[x_train["age"] < 18, "age"] = median_age
x_test.loc[x_test["age"] < 18, "age"] = median_age

x_train["age"] = x_train["age"].fillna(median_age)
x_test["age"] = x_test["age"].fillna(median_age)

# %%
# --- Clean department ---
department_mode = x_train["department"].mode()[0]
x_train["department"] = x_train["department"].fillna(department_mode)
x_test["department"] = x_test["department"].fillna(department_mode)

x_train["department"] = x_train["department"].str.strip().str.lower()
x_test["department"] = x_test["department"].str.strip().str.lower()

dept_map = {
    "suport": "support",
    "i.t.": "it",
    "human resources": "hr",
    "h.r.": "hr",
    "markting": "marketing",
    "ops": "operations"
}
x_train["department"] = x_train["department"].replace(dept_map)
x_test["department"] = x_test["department"].replace(dept_map)

# %%
# --- Clean city ---
city_mode = x_train["city"].mode()[0]
x_train["city"] = x_train["city"].fillna(city_mode)
x_test["city"] = x_test["city"].fillna(city_mode)

x_train["city"] = x_train["city"].str.strip().str.lower()
x_test["city"] = x_test["city"].str.strip().str.lower()

city_map = {"nyc": "new york"}
x_train["city"] = x_train["city"].replace(city_map)
x_test["city"] = x_test["city"].replace(city_map)

# %%
# --- Clean years_experience ---
median_exp = x_train["years_experience"].median()
x_train["years_experience"] = x_train["years_experience"].fillna(median_exp)
x_test["years_experience"] = x_test["years_experience"].fillna(median_exp)

invalid_train = x_train["years_experience"] > (x_train["age"] - 18)
x_train.loc[invalid_train, "years_experience"] = (x_train.loc[invalid_train, "age"] - 18).clip(lower=0)

invalid_test = x_test["years_experience"] > (x_test["age"] - 18)
x_test.loc[invalid_test, "years_experience"] = (x_test.loc[invalid_test, "age"] - 18).clip(lower=0)

# %%
# --- Clean remote_work ---
mode_remote = x_train["remote_work"].mode()[0]
x_train["remote_work"] = x_train["remote_work"].fillna(mode_remote)
x_test["remote_work"] = x_test["remote_work"].fillna(mode_remote)

# %%
# --- Clean performance_rating ---
median_perf = x_train["performance_rating"].median()
x_train["performance_rating"] = x_train["performance_rating"].fillna(median_perf)
x_test["performance_rating"] = x_test["performance_rating"].fillna(median_perf)

# %%
# --- Clean education ---
edu_mode = x_train["education"].mode()[0]
x_train["education"] = x_train["education"].fillna(edu_mode)
x_test["education"] = x_test["education"].fillna(edu_mode)

x_train["education"] = x_train["education"].str.strip().str.lower()
x_test["education"] = x_test["education"].str.strip().str.lower()

edu_map = {
    "bachelor": "bachelors"
}
x_train["education"] = x_train["education"].replace(edu_map)
x_test["education"] = x_test["education"].replace(edu_map)

# %%
# --- Clean gender ---
gender_mode = x_train["gender"].mode()[0]
x_train["gender"] = x_train["gender"].fillna(gender_mode)
x_test["gender"] = x_test["gender"].fillna(gender_mode)

x_train["gender"] = x_train["gender"].str.strip().str.lower()
x_test["gender"] = x_test["gender"].str.strip().str.lower()

gender_map = {
    "m": "male",
    "f": "female"
}
x_train["gender"] = x_train["gender"].replace(gender_map)
x_test["gender"] = x_test["gender"].replace(gender_map)

print(x_train["gender"].unique())  # confirm clean

# %%
# --- One-hot encode all categorical columns ---
cat_cols = ["department", "city", "education", "gender"]

x_train = pd.get_dummies(x_train, columns=cat_cols, drop_first=True).astype(int)
x_test = pd.get_dummies(x_test, columns=cat_cols, drop_first=True).astype(int)

x_train, x_test = x_train.align(x_test, join="left", axis=1, fill_value=0)

# %%
# --- Clean salary (target) ---
y_train = y_train.astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)
y_test = y_test.astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)

median_salary = y_train.median()
y_train[y_train > 200000] = median_salary
y_test[y_test > 200000] = median_salary

# %%
# --- Scale numeric columns ---
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
cols_to_scale = ["age", "years_experience", "performance_rating"]

x_train[cols_to_scale] = scaler.fit_transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler.transform(x_test[cols_to_scale])

# %%
# --- Train final model ---
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

model = LinearRegression()
model.fit(x_train, y_train)
pred = model.predict(x_test)

print("Linear Regression")
print("MAE :", mean_absolute_error(y_test, pred))
print("R2  :", r2_score(y_test, pred))

# %%
# --- Predict salary for a brand-new person ---
new_person = pd.DataFrame({
    "age": [35],
    "department": ["Engineering"],
    "city": ["Seattle"],
    "years_experience": [8],
    "remote_work": [1],
    "performance_rating": [4],
    "education": ["Masters"],
    "gender": ["Female"]
})

new_person["department"] = new_person["department"].str.strip().str.lower().replace(dept_map)
new_person["city"] = new_person["city"].str.strip().str.lower().replace(city_map)
new_person["education"] = new_person["education"].str.strip().str.lower().replace(edu_map)
new_person["gender"] = new_person["gender"].str.strip().str.lower().replace(gender_map)

new_person = pd.get_dummies(new_person, columns=["department", "city", "education", "gender"])
new_person = new_person.reindex(columns=x_train.columns, fill_value=0)

new_person[cols_to_scale] = scaler.transform(new_person[cols_to_scale])

predicted_salary = model.predict(new_person)
print(f"Predicted salary: ${predicted_salary[0]:,.2f}")

<StringArray>
['male', 'female']
Length: 2, dtype: str
Linear Regression
MAE : 5006.632232276731
R2  : 0.7937506783218594
Predicted salary: $96,184.30
